## Medical Reasoning LLM - Data Cleaning

### Objective

Prepare the medical reasoning dataset for downstream SFT and RLVR.

### Dataset

- Source: https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT
- Key fields:
    - Question
    - Complex_CoT
    - Response

### Pipeline

- Fetch Raw Dataset
- Train / Validation / Test Split
- Explore Dataset
- Deduplication
- Remove leading/trailing whitespace
- Format Validation
- Clean Dataset

## Fetch Raw Dataset

In [122]:
from datasets import load_dataset

dataset = load_dataset(
    "FreedomIntelligence/medical-o1-reasoning-SFT",
    "en",
)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['Question', 'Complex_CoT', 'Response'],
        num_rows: 19704
    })
})


In [28]:
!git -C /content/ml-ai-portfolio pull origin main

remote: Enumerating objects: 16, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 10 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 2.29 KiB | 1.14 MiB/s, done.
From https://github.com/xueqingnie/ml-ai-portfolio
 * branch            main       -> FETCH_HEAD
   b47d065..2e21db0  main       -> origin/main
Updating b47d065..2e21db0
Fast-forward
 .../notebooks/01_data_cleaning.ipynb               | 162 +++++++++++----------
 01-medical-reasoning-llm/src/data.py               |   4 +
 2 files changed, 91 insertions(+), 75 deletions(-)


In [29]:
!cat /content/ml-ai-portfolio/01-medical-reasoning-llm/src/data.py

from datasets import load_dataset

def fetch_hf_dataset(
    dataset_name: str,
    config: str = None,
    split: str = "train",
):
    """
    Load a dataset from Hugging Face Hub.
    
    Parameters:
    dataset_name : str
        Name of the dataset on Hugging Face.
    config : str, optional
        Dataset configuration, such as "en", "zh", "en_mix", or "zh_mix".
    split : str, default="train"
        Dataset split to load, such as "train", "test", or "validation".
    """
            
    return load_dataset(
        dataset_name,
        config,
        split=split,
    )


In [30]:
import importlib
import src.data

importlib.reload(src.data)

from src.data import fetch_hf_dataset

import inspect
print(inspect.signature(fetch_hf_dataset))

(dataset_name: str, config: str = None, split: str = 'train')


In [53]:
dataset = fetch_hf_dataset(
    "FreedomIntelligence/medical-o1-reasoning-SFT",
    config="en",
    split="train",
)

In [54]:
df = dataset.to_pandas()

In [91]:
df.head()

,Question,Complex_CoT,Response
0,Given the symptoms of sudden weakness in the l...,"Okay, let's see what's going on here. We've go...",The specific cardiac abnormality most likely t...
1,A 33-year-old woman is brought to the emergenc...,"Okay, let's figure out what's going on here. A...","In this scenario, the most likely anatomical s..."
2,A 61-year-old woman with a long history of inv...,"Okay, let's think about this step by step. The...",Cystometry in this case of stress urinary inco...
3,A 45-year-old man with a history of alcohol us...,"Alright, let’s break this down. We have a 45-y...",Considering the clinical presentation of sudde...
4,A 45-year-old man presents with symptoms inclu...,"Okay, so here's a 45-year-old guy who's experi...",Based on the clinical findings presented—wide-...


## Train / Validation / Test Split

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

## Data Exploration

In [57]:
print("Dataset shape:", df.shape)
print(df.columns.tolist())

Dataset shape: (19704, 3)
['Question', 'Complex_CoT', 'Response']


In [56]:
df.describe()

,Question,Complex_CoT,Response
count,19704,19704,19704
unique,19679,19704,19703
top,What is the name of the classification propose...,"Alright, let's think this through. The symptom...",D. The first statement is false and the second...
freq,3,1,2


In [58]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19704 entries, 0 to 19703
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Question     19704 non-null  object
 1   Complex_CoT  19704 non-null  object
 2   Response     19704 non-null  object
dtypes: object(3)
memory usage: 461.9+ KB


In [67]:
# Check for missing values

print(df.isnull().sum())

Question       0
Complex_CoT    0
Response       0
dtype: int64


In [100]:
# Check for empty strings

text_columns = ["Question", "Complex_CoT", "Response"]

for col in text_columns:
    empty_count = df[col].astype(str).str.strip().eq("").sum()
    print(f"{col}: {empty_count} empty strings")

Question: 0 empty strings
Complex_CoT: 0 empty strings
Response: 0 empty strings


In [101]:
# Check for duplicated rows and fields

print("Duplicated rows:", df.duplicated().sum())

for col in text_columns:
    duplicated_count = df[col].duplicated().sum()
    print(f"{col}: {duplicated_count} duplicated values")

Duplicated rows: 0
Question: 25 duplicated values
Complex_CoT: 0 duplicated values
Response: 1 duplicated values


In [102]:
# Check text length

df[text_columns].apply(
    lambda x: x.str.len()
).describe()

,Question,Complex_CoT,Response
count,19704.000000,19704.000000,19704.000000
mean,297.826634,1906.962850,638.603482
std,211.517985,408.177037,325.733624
min,51.000000,834.000000,4.000000
25%,163.000000,1621.000000,436.000000
50%,244.000000,1842.000000,570.000000
75%,358.000000,2129.000000,751.250000
max,2380.000000,5535.000000,3999.000000


In [105]:
df.loc[
    df["Question"].str.len().nsmallest(5).index,
    text_columns
]

,Question,Complex_CoT,Response
3215,In which conditions is sympathectomy not indic...,"Alright, let's think this through. Sympathecto...",Sympathectomy is not indicated for several con...
3871,Which bacterial species exhibit Dienes' phenom...,"Okay, let's think this through. I've heard abo...",Dienes' phenomenon is primarily associated wit...
5187,What functions or reflexes are lost in brain d...,"Alright, let's see what happens with brain dea...","In brain death, all functions and reflexes gov..."
14924,What are two key characteristics of laryngomal...,"Okay, let's think through this. What are we tr...",Two key characteristics of laryngomalacia are ...
17565,Which tests are used to assess vestibular func...,"Alright, so when it comes to understanding how...","To assess vestibular function, several special..."


In [104]:
df.loc[
    df["Complex_CoT"].str.len().nsmallest(5).index,
    text_columns
]

,Question,Complex_CoT,Response
19318,Calculate the relative risk for a population i...,"Alright, let's start by figuring out what this...",The relative risk for the population is 5. The...
19279,A 32-year-old male has a clean wound without l...,"Okay, let's think about tetanus prophylaxis fo...",For a 32-year-old male with a clean wound who ...
6389,"In a population of 100,000 under surveillance ...",I'm trying to figure out the Annual Parasite I...,To calculate the Annual Parasite Incidence (AP...
14307,A 59-year-old male presents with dimness of ne...,"Okay, let's think about this—this guy is 59 an...",The appropriate next step in managing this 59-...
15750,At what age is sexual development considered p...,"Alright, let's think about when kids usually s...","Sexual development, specifically breast and pu..."


In [106]:
df.loc[
    df["Response"].str.len().nsmallest(5).index,
    text_columns
]

,Question,Complex_CoT,Response
3329,Consider the following in a new born : Hea rat...,"Alright, let's figure out the Apgar score for ...",B. 3
13608,"Tasks that involve exposure blood, body fluids...","Okay, so when it comes to tasks that involve e...",A. I
16699,Consider the following statements in respect o...,Let's take a look at these statements one at a...,D. 3
19242,Father has a blood group B : Mother has AB : C...,"Okay, let's figure this out. We know blood gro...",A. 0
2497,Induction of labor by amniotomy can lead to th...,"Okay, so I'm thinking about the risks involved...",A. ad


In [81]:
df.loc[
    df["Question"].str.len().nlargest(5).index,
    ["Question", "Complex_CoT", "Response"]
]

,Question,Complex_CoT,Response
6170,A 59-year-old male with a history of aortic st...,"Alright, let's think about this situation with...",The patient's presentation is highly suggestiv...
8608,BACKGROUND:\nAldosterone blockade reduces mort...,"Alright, let's see what's happening with this ...",The relative risk reduction (RRR) in all-cause...
14569,"A 40-year-old man presents with a rash, oral l...","Alright, we've got a 40-year-old man dealing w...",To confirm the most likely diagnosis of Steven...
15439,"A 65-year-old woman, with end-stage renal dise...","Okay, let's think this through. We have a 65-y...",The symptoms and laboratory findings you have ...
19042,Please refer to the summary above to answer th...,"Alright, let’s see what's going on with this p...",The information provided points towards a diag...


In [107]:
df.loc[
    df["Complex_CoT"].str.len().nlargest(5).index,
    text_columns
]

,Question,Complex_CoT,Response
15353,"Based on this patient's clinical presentation,...","Alright, let's figure out what's causing these...",Based on the detailed analysis of the patient'...
11307,"Based on the microscopic images provided, what...","Alright, let's take a closer look at this brea...",Based on the detailed assessment and review of...
4358,Based on the microscopic examination of the bi...,"Alright, let's take a closer look at this biop...",Based on the microscopic examination described...
14743,Based on the microscopic examination of the pr...,"Alright, let's dive into these microscopic ima...","Based on the microscopic examination, the most..."
3896,What impact does successful whole-organ pancre...,"Alright, let's dive into this topic. So, when ...",A successful whole-organ pancreas transplantat...


In [108]:
df.loc[
    df["Response"].str.len().nlargest(5).index,
    text_columns
]

,Question,Complex_CoT,Response
10362,Which natural elements are commonly recognized...,When thinking about natural elements with anti...,Many natural elements are recognized for their...
11926,Explain the process and chromosomal coding inv...,"Okay, so immunoglobulins, or antibodies, are r...","The formation of immunoglobulins, or antibodie..."
16770,In the context of treating inflammation in rhe...,"Alright, let's think about how steroids work t...","Steroids, specifically glucocorticoids, are hi..."
8340,In the case of a 4-year-old boy with a Strepto...,"Okay, so first things first. The kid's body is...",In the case of a 4-year-old boy with a Strepto...
1344,What is the most appropriate way for a physici...,"Okay, so we're dealing with a 15-year-old girl...",To obtain a more in-depth social history from ...


In [109]:
for col in text_columns:
    print(f"\n===== {col} =====")
    print(df[col].head(3).to_list())


===== Question =====
['Given the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?', 'A 33-year-old woman is brought to the emergency department 15 minutes after being stabbed in the chest with a screwdriver. Given her vital signs of pulse 110/min, respirations 22/min, and blood pressure 90/65 mm Hg, along with the presence of a 5-cm deep stab wound at the upper border of the 8th rib in the left midaxillary line, which anatomical structure in her chest is most likely to be injured?', 'A 61-year-old woman with a long history of involuntary urine loss during activities like coughing or sneezing but no leakage at night undergoes a gynecological exam and Q-tip test. Based on these findings, what would cystometry most likely reveal about her residual volume and detrusor contractions?']

## Deduplication

In [93]:
duplicated_questions = df[df["Question"].duplicated(keep=False)]

duplicated_questions.sort_values("Question")

,Question,Complex_CoT,Response
4527,"After a laparoscopic cholecystectomy, at which...","Okay, so we're talking about where a biliary s...","After a laparoscopic cholecystectomy, a biliar..."
14702,"After a laparoscopic cholecystectomy, at which...","Alright, so we're looking at what happens afte...","After a laparoscopic cholecystectomy, a biliar..."
12740,At what age does a child typically begin to si...,Let's think about when a child starts doing ce...,"Typically, a child begins to sit with support,..."
6273,At what age does a child typically begin to si...,So when do kids start to sit with a little hel...,"Typically, a child begins to sit with support,..."
14949,At what age does a child typically begin to si...,"Okay, let's think about when babies typically ...",Babies typically begin to sit with support aro...
2491,At what left atrial pressure does pulmonary ed...,"Alright, so we're trying to figure out when pu...",Pulmonary edema generally begins to appear whe...
3724,At what left atrial pressure does pulmonary ed...,"So, let's think about pulmonary edema for a mo...",Pulmonary edema typically begins to appear whe...
13144,In a 3-week-old child presenting with an abdom...,A 3-week-old baby with an abdominal mass—what ...,In a 3-week-old child presenting with an abdom...
5361,In a 3-week-old child presenting with an abdom...,Let's think about what could cause an abdomina...,In a 3-week-old child presenting with an abdom...
5620,To which layers of the lateral geniculate nucl...,"Okay, so let's think about the visual pathway ...",Fibers from the contralateral nasal hemiretina...


In [111]:
question_groups = duplicated_questions.groupby("Question")

In [112]:
question_groups["Complex_CoT"].nunique()

,Complex_CoT
Question,
"After a laparoscopic cholecystectomy, at which part of the common bile duct does a biliary stricture most commonly develop?",2
"At what age does a child typically begin to sit with support, transfer objects from one hand to another, and speak monosyllabic babbles?",3
At what left atrial pressure does pulmonary edema generally begin to appear following acute failure of the left ventricle in humans?,2
"In a 3-week-old child presenting with an abdominal mass, what is the most common cause of this presentation?",2
To which layers of the lateral geniculate nucleus do the fibers from the contralateral nasal hemiretina project?,2
What concept is associated with emotional valence and is most likely to be influenced by motivation?,2
What condition is indicated by an X-ray showing an air column between a soft tissue mass and the posterior wall of the nasopharynx?,2
What is the best imaging technique for differentiating the recurrence of a brain tumor from radiation therapy-induced necrosis?,2
What is the immediate treatment modality for a patient presenting with acute anterior wall infarction and hypotension?,2


In [113]:
question_groups["Response"].nunique()

,Response
Question,
"After a laparoscopic cholecystectomy, at which part of the common bile duct does a biliary stricture most commonly develop?",2
"At what age does a child typically begin to sit with support, transfer objects from one hand to another, and speak monosyllabic babbles?",3
At what left atrial pressure does pulmonary edema generally begin to appear following acute failure of the left ventricle in humans?,2
"In a 3-week-old child presenting with an abdominal mass, what is the most common cause of this presentation?",2
To which layers of the lateral geniculate nucleus do the fibers from the contralateral nasal hemiretina project?,2
What concept is associated with emotional valence and is most likely to be influenced by motivation?,2
What condition is indicated by an X-ray showing an air column between a soft tissue mass and the posterior wall of the nasopharynx?,2
What is the best imaging technique for differentiating the recurrence of a brain tumor from radiation therapy-induced necrosis?,2
What is the immediate treatment modality for a patient presenting with acute anterior wall infarction and hypotension?,2


In [114]:
question_groups.agg(
    response_count=("Response", "nunique"),
    cot_count=("Complex_CoT", "nunique")
)

,response_count,cot_count
Question,,
"After a laparoscopic cholecystectomy, at which part of the common bile duct does a biliary stricture most commonly develop?",2,2
"At what age does a child typically begin to sit with support, transfer objects from one hand to another, and speak monosyllabic babbles?",3,3
At what left atrial pressure does pulmonary edema generally begin to appear following acute failure of the left ventricle in humans?,2,2
"In a 3-week-old child presenting with an abdominal mass, what is the most common cause of this presentation?",2,2
To which layers of the lateral geniculate nucleus do the fibers from the contralateral nasal hemiretina project?,2,2
What concept is associated with emotional valence and is most likely to be influenced by motivation?,2,2
What condition is indicated by an X-ray showing an air column between a soft tissue mass and the posterior wall of the nasopharynx?,2,2
What is the best imaging technique for differentiating the recurrence of a brain tumor from radiation therapy-induced necrosis?,2,2
What is the immediate treatment modality for a patient presenting with acute anterior wall infarction and hypotension?,2,2


In [115]:
question = (question_groups.size().sort_values(ascending=False).index[0])

question_groups.get_group(question)[text_columns]

,Question,Complex_CoT,Response
6273,At what age does a child typically begin to si...,So when do kids start to sit with a little hel...,"Typically, a child begins to sit with support,..."
12740,At what age does a child typically begin to si...,Let's think about when a child starts doing ce...,"Typically, a child begins to sit with support,..."
14949,At what age does a child typically begin to si...,"Okay, let's think about when babies typically ...",Babies typically begin to sit with support aro...


## Format Validation

In [116]:
# Check for leading/trailing whitespace

for col in text_columns:
    leading = df[col].str.startswith(" ").sum()
    trailing = df[col].str.endswith(" ").sum()

    print(f"{col}:")
    print(f"  Leading spaces: {leading}")
    print(f"  Trailing spaces: {trailing}")

Question:
  Leading spaces: 0
  Trailing spaces: 0
Complex_CoT:
  Leading spaces: 0
  Trailing spaces: 0
Response:
  Leading spaces: 0
  Trailing spaces: 0


In [117]:
# Strip leading/trailing whitespace

for col in text_columns:
    df[col] = df[col].str.strip()

In [119]:
# Re-check for empty strings after stripping leading/trailing whitespace

for col in text_columns:
    empty_count = df[col].str.strip().eq("").sum()
    print(f"{col}: {empty_count} empty strings")

Question: 0 empty strings
Complex_CoT: 0 empty strings
Response: 0 empty strings


In [120]:
# Check for placeholders

placeholders = [
    "N/A",
    "NA",
    "None",
    "null",
    "NULL",
    "unknown",
    "Unknown",
]

for col in text_columns:
    matches = df[col].isin(placeholders).sum()
    print(f"{col}: {matches} placeholders")

Question: 0 placeholders
Complex_CoT: 0 placeholders
Response: 0 placeholders
